# Enhancer & promoters 

This notebook downloads and transforms the [Epiraction](https://www.biorxiv.org/content/10.1101/2025.02.18.638885v1) dataset from ENCODE, which corresponts to enhancer-promoter interactions.


To download and prepare the data you need following:
- `uvx`
- `bedtools`
- `bgzip` and `tabix` (from `htslib`)
- `parallel` (GNU parallel)

The script `scripts/download-epiraction.sh` automates the download and preparation of the data. It downloads the data into the `data/25.06/epiraction` directory, unzips, sorts, bgzips, and indexes the files. The outputed files are in `BED` format, block gzipped and indexed with `tabix`.

## Data Download

In [9]:
%%bash

(cd ../../ && ./scripts/download-epiraction.sh)


/home/mindos/Projects/OpenTargets/gentropy-manuscript
Starting
.
Creating data directory at ./data/25.06/EPIraction/raw
Changing to data directory



real	1m40,546s
user	0m16,008s
sys	0m10,884s


Download complete.
Unzipping



real	0m3,038s
user	0m23,286s
sys	0m11,637s


Sorting


...

Indexing



real	0m3,059s
user	0m38,423s
sys	0m1,378s


Cleaning up intermediate files



real	0m2,104s
user	0m0,001s
sys	0m2,097s


Completed.


## Transform to parquet

The input datasets are transformed from raw data files downloaded from ENCODE to parquet format for faster access during the analysis.

In [10]:
from gentropy import Session


In [ ]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"}, use_enhanced_bgzip_codec=True)

biosample_index_path = "../../data/25.06/output/biosample"
target_index_path = "../../data/25.06/output/target"
epiraction_raw_data = "../../data/25.06/EPIraction/raw/*.gz"
epiraction_processed = "../../data/25.06/EPIraction/processed"
spark = session.spark
spark


In [4]:
epiraction_raw = (
    spark.read.csv(
        epiraction_raw_data,
        header=True,
        inferSchema=True,
        sep="\t",
        enforceSchema=False,
    )
    .repartition(10)
    .write.mode("overwrite")
    .parquet(epiraction_processed)
)


In [5]:
epiraction = spark.read.parquet(epiraction_processed)
print(f"Number of Intervals  - {epiraction.count():,}")


Number of Intervals  - 62,301,458


### Delete raw data to save space

In [6]:
%%bash
rm -rf ../../data/25.06/EPIraction/raw/


In [ ]:
spark.stop()
